In [1]:
"""Evaluate forecast predictions: MAE, RMSE, MAPE and horizon-wise errors.

Input CSV columns: Actual, Predicted, Target, Horizon, Model
Run: python evaluation.py --input forecast_predictions.csv
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:

REQUIRED = ["Actual", "Predicted", "Target", "Horizon", "Model"]



In [3]:

def mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(predicted) & (actual != 0)
    if not valid.any():
        return np.nan
    return float(np.mean(np.abs((actual[valid] - predicted[valid]) / actual[valid])) * 100)




In [4]:
def evaluate_forecasts(df):
    aliases = {
        "actual": "Actual",
        "predicted": "Predicted",
        "target": "Target",
        "horizon_minutes": "Horizon",
        "model": "Model",
    }
    df = df.rename(columns={c: aliases[c] for c in df.columns if c in aliases and aliases[c] not in df.columns})
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}. Required: {REQUIRED}")

    df = df.copy()
    for col in ["Actual", "Predicted"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=REQUIRED)
    if df.empty:
        raise ValueError("No valid forecast rows found.")

    def metrics(group):
        y, p = group["Actual"].to_numpy(), group["Predicted"].to_numpy()
        return pd.Series({
            "Samples": len(group),
            "MAE": mean_absolute_error(y, p),
            "RMSE": np.sqrt(mean_squared_error(y, p)),
            "MAPE (%)": mape(y, p),
        })

    overall = df.groupby(["Target", "Model"]).apply(metrics).reset_index()
    by_horizon = df.groupby(["Target", "Horizon", "Model"]).apply(metrics).reset_index()
    return overall.sort_values(["Target", "MAE"]), by_horizon.sort_values(
        ["Target", "Horizon", "MAE"]
    )



In [5]:

def main():
    parser = argparse.ArgumentParser()
    default_candidates = [
        Path.cwd() / "model_outputs" / "test_predictions.csv",
        Path.cwd() / "dataProcess" / "model_outputs" / "test_predictions.csv",
    ]
    default_input = next(
        (candidate for candidate in default_candidates if candidate.is_file()),
        default_candidates[0],
    )
    parser.add_argument("--input", default=str(default_input), help="Forecast predictions CSV")
    parser.add_argument("--output-dir", default="evaluation_results")
    args, extra_args = parser.parse_known_args()
    if extra_args:
        is_kernel_args = (
            len(extra_args) == 1 and extra_args[0].startswith("--f=")
        ) or (
            len(extra_args) == 2 and extra_args[0] in {"-f", "--f"}
        )
        if not is_kernel_args:
            parser.error(f"unrecognized arguments: {' '.join(extra_args)}")

    source = Path(args.input)
    if not source.is_file():
        raise FileNotFoundError(f"CSV not found: {source}")

    overall, horizon = evaluate_forecasts(pd.read_csv(source))
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    overall.to_csv(output / "evaluation_metrics.csv", index=False)
    horizon.to_csv(output / "horizon_wise_metrics.csv", index=False)
    print("\\nOverall evaluation metrics:\\n", overall.to_string(index=False))
    print("\\nHorizon-wise metrics:\\n", horizon.to_string(index=False))
    print(f"\\nSaved results to: {output.resolve()}")


if __name__ == "__main__":
    main()


\nOverall evaluation metrics:\n           Target                        Model  Samples        MAE       RMSE    MAPE (%)
Redemption Count                      XGBoost   5130.0  39.866050  67.842335   46.247557
Redemption Count                 RandomForest   5130.0  39.908087  67.946792   54.652105
Redemption Count             GradientBoosting   5130.0  42.506164  69.391857   59.570045
Redemption Count             LinearRegression   5130.0  49.754774  77.425256  108.905184
Redemption Count MovingAverage4TrainingValues   5130.0 283.246101 322.266155 1285.228124
Redemption Count       NaiveLastTrainingValue   5130.0 304.644639 346.900468 1374.981747
     Sales Count                 RandomForest   5130.0  33.442559  52.417818   61.763382
     Sales Count                      XGBoost   5130.0  33.702287  52.713477   67.582528
     Sales Count             GradientBoosting   5130.0  35.454265  55.369316   83.182546
     Sales Count             LinearRegression   5130.0  40.906443  59.426160  